# Phase 3: Neural Re-ranking & Evaluation Pipeline
**Team Member 3: Neural Re-ranking & Evaluation Engineer**

This notebook implements the final stage of the CISI Information Retrieval pipeline:
1. **Input**: Top-100 candidate documents from Phase 2 (Hybrid/Dense/BM25 retrieval)
2. **Models**: 
    - Cross-Encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
    - Point-wise Re-ranker (`castorini/monot5-base-msmarco`)
3. **Evaluation**: Compute MRR@10, P@10, NDCG, and Recall using `ranx` library.
4. **Analysis**: Provide detailed query-by-query performance insights.

In [ ]:
import sys
import os
import json
import torch
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

# Add project root to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import project modules
from reranking.cross_encoder_reranker import CrossEncoderReranker
from reranking.monot5_reranker import MonoT5Reranker
from ranx import Qrels, Run, evaluate
from evaluation.eval_pipeline import evaluate_pipeline
from analysis.analyze_reranking_results import generate_report

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Dataset and Top-100 Candidates
Load the queries, corpus, and the best available top-100 initial retrieval results.

In [ ]:
data_dir = Path('../data')

with open(data_dir / 'queries.json', 'r') as f:
    queries_data = json.load(f)
    queries = {str(q['query_id']): q['text'] for q in queries_data}

with open(data_dir / 'corpus.json', 'r') as f:
    corpus_data = json.load(f)
    corpus = {str(d['doc_id']): f"{d.get('title', '')} {d.get('text', '')}".strip() for d in corpus_data}

with open(data_dir / 'qrels.json', 'r') as f:
    qrels = json.load(f)

# Fallback mechanism: Try Hybrid -> Dense -> BM25
candidates_file = None
for candidate_name in ['hybrid_top100.json', 'dense_top100.json', 'bm25_top100.json']:
    if (data_dir / candidate_name).exists():
        candidates_file = data_dir / candidate_name
        break

if not candidates_file:
    raise FileNotFoundError("No top-100 candidate files found in data/")

print(f"Loading initial retrieval candidates from: {candidates_file.name}")
with open(candidates_file, 'r') as f:
    top100_candidates = json.load(f)

print(f"Loaded {len(queries)} queries, {len(corpus)} documents.")
print(f"Loaded {len(top100_candidates)} queries with candidates.")

## 2. Initialize Models
Loading Cross-Encoder and MonoT5 models to GPU.

In [ ]:
print("Initializing Cross-Encoder...")
ce_reranker = CrossEncoderReranker()

print("\nInitializing MonoT5...")
monot5_reranker = MonoT5Reranker()

## 3. Run Neural Re-ranking Pipeline
Re-rank the top-100 candidates using both models. Note: This might take a few minutes depending on GPU.

In [ ]:
ce_results = {}
monot5_results = {}

for qid in tqdm(top100_candidates.keys(), desc="Re-ranking Queries"):
    query_text = queries[str(qid)]
    
    # Docs are list of dicts: [{'doc_id': doc1, 'score': score1}, ...]
    doc_ids = [str(d['doc_id']) for d in top100_candidates[qid]]
    doc_texts = [corpus[d_id] for d_id in doc_ids]
    
    # Cross-Encoder
    ce_ranked = ce_reranker.rerank(query_text, doc_texts, doc_ids)
    ce_results[qid] = ce_ranked
    
    # MonoT5
    mt5_ranked = monot5_reranker.rerank(query_text, doc_texts, doc_ids)
    monot5_results[qid] = mt5_ranked

print("\nRe-ranking complete. Saving results...")
with open(data_dir / 'ce_reranked.json', 'w') as f:
    json.dump(ce_results, f, indent=2)
with open(data_dir / 'monot5_reranked.json', 'w') as f:
    json.dump(monot5_results, f, indent=2)
print("Results saved successfully.")

## 4. Evaluation via RANX 
Computing exact metrics comparing BM25/Dense/Hybrid vs Neural Re-rankers.

In [ ]:
# Import the existing evaluation pipeline function to run it and save metrics
print("Running complete evaluation pipeline on all available models...")
evaluate_pipeline()

# Load the saved metrics table to display
metrics_path = Path('../reports/final_metrics_table.json')
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print("\n========= METRICS SUMMARY =========")
    print(pd.DataFrame(metrics).T.round(4).to_markdown())

## 5. Detailed Analysis
Leveraging the script `analyze_reranking_results.py` to identify detailed improvements and degradations across all queries.

In [ ]:
print("Generating complete analysis report...\n")

# Run the detailed report generation directly
report = generate_report(qrels, queries, top100_candidates, ce_results, monot5_results)

# Display a preview of the report (First 50 lines for brevity)
print("\n--- Report Preview (First 50 lines) ---")
print('\n'.join(report.split('\n')[:50]))